# Clean Task2 acquisition follow-up (exploratory)

This notebook runs exactly two **clean** configurations from the same verified Task1 best checkpoint and 25,000-row, training-only frozen replay. Historical test results have already been viewed, so this is exploratory. It does not retrain Task1 or adjust the quarantine threshold. Do not infer eight-class success from aggregate accuracy: historical B had 97.70% combined accuracy but only 2.35% new-class accuracy and zero Infiltration recall.

A and B each use seed 42, LR 0.0003, batch 256, 141,152 draws/epoch, up to 12 epochs, patience four, validation macro-F1 checkpoint selection, and quotas: 40% old replay (original Task1 training proportions), 40% Infiltration, 10% Webattack, 10% Portscan. A uses eight-class CE. B uses the same CE plus 0.5 times temperature-2, batch-normalized T² KL from the frozen Task1 teacher on old replay rows and old logits only. These values are fixed **before** execution. Compared with historical B, sampling, exposure and LR change together; only A versus B isolates KD.

Attach the complete prepared arrays folder, complete Task1 run folder, and intact clean-retention evidence ZIP or extracted evidence folder. Enable **GPU** and **Internet** in Kaggle. Generated checkpoints and bulk metrics must remain outside Git. If Kaggle stops, save the whole output folder as a private Kaggle dataset and resume with `RESUME_RUN_DIR`; keep the same GPU/runtime for exact resume.


In [ ]:
import pathlib, subprocess
SOURCE_SHA = '095c7b639d54eaf196a3bb324ce147a7c43c12e0'
REPO = pathlib.Path('/kaggle/working/poison-resilient-continual-nids')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/DevChi1l/poison-resilient-continual-nids.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', 'origin', 'feature/unified-novelty'], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() == SOURCE_SHA
print('Verified source:', SOURCE_SHA)


In [ ]:
import importlib.util, pathlib, sys
sys.path.insert(0, str(REPO))
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU and restart'
PREPARED_DIR = pathlib.Path('/kaggle/input/notebooks/dvdhere/preparation/review2_large_prepare_20261002T223303_224539Z')
TASK1_RUN_DIR = pathlib.Path('/kaggle/input/notebooks/dvdhere/full-run-task1/full_task1_20261003T074354_763068Z')
RETENTION_RESULTS_DIR = pathlib.Path('/kaggle/input/datasets/dvdhere/results-task2/results_ret/task2_clean_retention_study')
RETENTION_RESULTS_ZIP = None  # Set to an intact ZIP instead, and set DIR=None.
OUTPUT_DIR = pathlib.Path('/kaggle/working/task2_clean_acquisition_study')
RESUME_RUN_DIR = None  # Set to an existing complete output folder to resume.
MAX_EPOCHS_THIS_CALL = None  # Optional epoch-boundary pause, e.g. 1.
assert PREPARED_DIR.is_dir() and TASK1_RUN_DIR.is_dir()
assert (RETENTION_RESULTS_DIR is None) != (RETENTION_RESULTS_ZIP is None)
assert RETENTION_RESULTS_DIR.is_dir() if RETENTION_RESULTS_DIR is not None else RETENTION_RESULTS_ZIP.is_file()
print('GPU:', torch.cuda.get_device_name(0))
print('Prepared:', PREPARED_DIR, 'Task1:', TASK1_RUN_DIR)
print('Retention evidence:', RETENTION_RESULTS_DIR or RETENTION_RESULTS_ZIP)


In [ ]:
# Fast synthetic checks only. No local or Kaggle research-data training here.
subprocess.run([sys.executable, '-m', 'pytest', '-q',
    'tests/test_task2_acquisition.py', 'tests/test_task2_replay.py',
    'tests/test_full_disk_training.py'], cwd=REPO, check=True)


In [ ]:
command = [sys.executable, 'scripts/run_task2_acquisition.py',
    '--prepared-dir', str(PREPARED_DIR), '--task1-run-dir', str(TASK1_RUN_DIR)]
if RETENTION_RESULTS_DIR is not None:
    command += ['--retention-results-dir', str(RETENTION_RESULTS_DIR)]
else:
    command += ['--retention-results-zip', str(RETENTION_RESULTS_ZIP)]
command += (['--resume-run-dir', str(RESUME_RUN_DIR)] if RESUME_RUN_DIR is not None
            else ['--output-dir', str(OUTPUT_DIR)])
if MAX_EPOCHS_THIS_CALL is not None:
    command += ['--max-epochs-this-call', str(MAX_EPOCHS_THIS_CALL)]
print('Running:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)
print('Read protocol.json, arm histories/summaries, validation_metrics.json, selection_lock.json, and selected_test_summary.json before any claim.')
